In [29]:
# # ==== SETUP: Download data files ====
# # Run this cell ONCE to download the data folder from Google Drive.
# # After it finishes, you can skip this cell in future runs.

# import subprocess, sys, os
# subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
# import gdown
# GOOGLE_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1ei8-m-N2vJHj2y7LHA7QYEC7FT2G-qDN?usp=sharing"

# output_dir = "data"
# os.makedirs(output_dir, exist_ok=True)

# gdown.download_folder(GOOGLE_DRIVE_FOLDER_URL, output=output_dir, quiet=False)

# print("Done! All data files downloaded to the 'data/' folder.")

In [30]:
import pandas as pd
#1: data colleciton
# Load all 4 files
ff_factors = pd.read_csv("data/F-F_Research_Data_Factors.csv", skiprows=3)
ff_mom = pd.read_csv("data/F-F_Momentum_Factor.csv", skiprows=13)
sic_mapping = pd.read_excel("data/SIC_49_Industry.xlsx")
oap_factors = pd.read_csv("data/signed_predictors_dl_wide.csv")
crsp = pd.read_csv("data/crsp.csv")

# Print heads
print("=== FF 3 Factors ===")
print(ff_factors.head())
print(f"\nShape: {ff_factors.shape}")

print("\n=== FF Momentum ===")
print(ff_mom.head())
print(f"\nShape: {ff_mom.shape}")

print("\n=== SIC 49 Industry Mapping ===")
print(sic_mapping.head())
print(f"\nShape: {sic_mapping.shape}")

print("\n=== OAP Factors ===")
print(oap_factors.head())
print(f"\nShape: {oap_factors.shape}")

print("\n=== CRSP ===")
print(crsp.head())
print(f"\nShape: {crsp.shape}")

/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_82912/4241882157.py:8: DtypeWarning: Columns (0: SICCD) have mixed types. Specify dtype option on import or set low_memory=False.
  crsp = pd.read_csv("data/crsp.csv")


=== FF 3 Factors ===
  Unnamed: 0   Mkt-RF      SMB      HML       RF
0     192607     2.89    -2.55    -2.39     0.22
1     192608     2.64    -1.14     3.81     0.25
2     192609     0.38    -1.36     0.05     0.23
3     192610    -3.27    -0.14     0.82     0.32
4     192611     2.54    -0.11    -0.61     0.31

Shape: (1298, 5)

=== FF Momentum ===
  Unnamed: 0      Mom
0     192701     0.57
1     192702    -1.50
2     192703     3.52
3     192704     4.36
4     192705     2.78

Shape: (1293, 2)

=== SIC 49 Industry Mapping ===
   Industry  SIC_start  SIC_end Industry_name
0         1        100      199         Agric
1         1        200      299         Agric
2         1        700      799         Agric
3         1        910      919         Agric
4         1       2048     2048         Agric

Shape: (598, 4)

=== OAP Factors ===
   permno  yyyymm  AM  AOP  AbnormalAccruals  Accruals  AccrualsBM  Activism1  \
0   10000  198601 NaN  NaN               NaN       NaN         NaN  

In [31]:
#2: Data cleaning
import pandas as pd
import numpy as np

#config the 5 and 100 thresholds are given per assignemnt
DATA_DIR = "data"
OUTPUT_DIR = "cleaned_data"
START_YEAR = 1985
END_YEAR = 2023
MIN_PRICE = 5.0
MIN_MARKET_CAP = 100

#CRSP cleaning:
before_rows = len(crsp)
crsp.columns = crsp.columns.str.upper()
crsp['DATE'] = pd.to_datetime(crsp['DATE'])
crsp['YEAR'] = crsp['DATE'].dt.year
crsp['MONTH'] = crsp['DATE'].dt.month
crsp['YYYYMM'] = crsp['YEAR'] * 100 + crsp['MONTH']
crsp = crsp[crsp['SHRCD'].isin([10, 11])] 
crsp = crsp[crsp['EXCHCD'].isin([1, 2, 3])]
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp = crsp[crsp['RET'].notna()]
crsp = crsp[crsp['RET'] > -1.0] 
crsp['PRC_ABS'] = crsp['PRC'].abs()
crsp['MARKET_CAP'] = crsp['PRC_ABS'] * crsp['SHROUT'] 
crsp = crsp[crsp['PRC_ABS'] >= MIN_PRICE] 
crsp = crsp[crsp['MARKET_CAP'] >= MIN_MARKET_CAP * 1000] 
crsp = crsp[(crsp['YEAR'] >= START_YEAR) & (crsp['YEAR'] <= END_YEAR)]
 
print(f"CRSP cleaned: {len(crsp):,} rows. (Removed {before_rows - len(crsp):,} rows)")


CRSP cleaned: 1,270,055 rows. (Removed 2,420,213 rows)


In [32]:
# Vectorized FF49 mapping: interval lookup instead of per-row Python loop
sic_mapping_sorted = sic_mapping.sort_values('SIC_start').reset_index(drop=True)
intervals = pd.IntervalIndex.from_arrays(
    sic_mapping_sorted['SIC_start'],
    sic_mapping_sorted['SIC_end'],
    closed='both',
)
industry_values = sic_mapping_sorted['Industry'].to_numpy()

sic_numeric = pd.to_numeric(crsp['SICCD'], errors='coerce')
idx = intervals.get_indexer(sic_numeric.fillna(-1).to_numpy())
crsp['FF49'] = np.where(idx >= 0, industry_values[idx], 49).astype('int16')

# Vectorized annual compounding via log1p/expm1 + groupby.sum (C-level, no Python apply)
log1p_ret = np.log1p(crsp['RET'].to_numpy())
annual_returns = (
    pd.DataFrame({
        'PERMNO': crsp['PERMNO'].to_numpy(),
        'YEAR': crsp['YEAR'].to_numpy(),
        'LOG1P_RET': log1p_ret,
    })
    .groupby(['PERMNO', 'YEAR'], sort=False, as_index=False)['LOG1P_RET']
    .sum()
)
annual_returns['RET_ANNUAL'] = np.expm1(annual_returns['LOG1P_RET'])
annual_returns = annual_returns[['PERMNO', 'YEAR', 'RET_ANNUAL']]

print(f"Annual returns: {len(annual_returns):,} stock-years")

oap_rows_before = len(oap_factors)
dec_crsp = crsp.loc[crsp['MONTH'] == 12, ['PERMNO', 'YEAR', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']].copy()
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1
print(f"oap_factors December: {len(oap_factors_dec):,} rows. Before: {oap_rows_before:,} rows.")


Annual returns: 124,875 stock-years


/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_82912/3489364595.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
/var/folders/dv/8r8942l51v104x9yl5sywbtm0000gn/T/ipykernel_82912/3489364595.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oap_factors['MONTH'] = oap_factors['yyyymm'] % 100


oap_factors December: 454,907 rows. Before: 5,416,424 rows.


In [33]:
oap_factors['YEAR'] = oap_factors['yyyymm'] // 100
oap_factors['MONTH'] = oap_factors['yyyymm'] % 100
oap_factors_dec = oap_factors[oap_factors['MONTH'] == 12].copy()
oap_factors_dec['RETURN_YEAR'] = oap_factors_dec['YEAR'] + 1  # Dec T factors predict T+1 returns
 
print(f"OAP December: {len(oap_factors_dec):,} rows")

merged = pd.merge(
    oap_factors_dec,
    annual_returns,
    left_on=['permno', 'RETURN_YEAR'],
    right_on=['PERMNO', 'YEAR'],
    how='inner'
)
 
# Merge with December CRSP data for FF49
merged = pd.merge(
    merged,
    dec_crsp,
    left_on=['permno', 'YEAR_x'],
    right_on=['PERMNO', 'YEAR'],
    how='left'
)
merged.head()


OAP December: 454,907 rows


,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,...,RETURN_YEAR,PERMNO_x,YEAR_y,RET_ANNUAL,PERMNO_y,YEAR,PRC_ABS,MARKET_CAP,SICCD,FF49
0,10001,201212,2.055205,0.013963,0.014669,0.033528,NaN,NaN,NaN,NaN,...,2013,10001,2013,-0.014836,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,201312,2.078684,NaN,-0.032669,0.018642,NaN,NaN,NaN,NaN,...,2014,10001,2014,0.225805,NaN,NaN,NaN,NaN,NaN,NaN
2,10001,201412,1.762830,0.042705,-0.005713,0.034970,NaN,NaN,NaN,NaN,...,2015,10001,2015,-0.065084,10001.0,2014.0,11.02,115577.76,4925.0,31.0
3,10001,201512,2.734447,NaN,0.001784,0.018868,NaN,NaN,NaN,NaN,...,2016,10001,2016,0.656020,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,201612,1.497349,NaN,-0.039445,0.063455,NaN,NaN,NaN,NaN,...,2017,10001,2017,0.043988,10001.0,2016.0,12.55,132026.00,4925.0,31.0


In [34]:
ff_factors.columns = ['YYYYMM', 'MKT_RF', 'SMB', 'HML', 'RF']
ff_factors = ff_factors[ff_factors['YYYYMM'].astype(str).str.len() == 6]
ff_factors['YYYYMM'] = ff_factors['YYYYMM'].astype(int)
ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']] = ff_factors[['MKT_RF', 'SMB', 'HML', 'RF']].apply(pd.to_numeric, errors='coerce') / 100
 
ff_mom.columns = ['YYYYMM', 'UMD']
ff_mom = ff_mom[ff_mom['YYYYMM'].astype(str).str.len() == 6]
ff_mom['YYYYMM'] = ff_mom['YYYYMM'].astype(int)
ff_mom['UMD'] = pd.to_numeric(ff_mom['UMD'], errors='coerce') / 100
 
ff_all = pd.merge(ff_factors, ff_mom, on='YYYYMM', how='inner')
ff_all.head()

,YYYYMM,MKT_RF,SMB,HML,RF,UMD
0,192701,-0.0005,-0.0032,0.0458,0.0025,0.0057
1,192702,0.0417,0.0007,0.0272,0.0026,-0.0150
2,192703,0.0014,-0.0177,-0.0238,0.0030,0.0352
3,192704,0.0047,0.0039,0.0065,0.0025,0.0436
4,192705,0.0545,0.0155,0.0480,0.0030,0.0278


In [35]:
id_cols = ['permno', 'yyyymm', 'YEAR', 'MONTH', 'RETURN_YEAR', 'PERMNO', 'YEAR_x', 'YEAR_y',
           'RET_ANNUAL', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']
factor_cols = [c for c in merged.columns if c not in id_cols]
 
missing_pct = (merged[factor_cols].isna().sum() / len(merged) * 100).sort_values()
print("Missing data percentage per factor:")
print(missing_pct)

Missing data percentage per factor:
PERMNO_x                 0.000000
DivInit                  0.032327
DivOmit                  0.032327
MaxRet                   0.213530
ExchSwitch               0.239051
                          ...    
AccrualsBM              95.106680
Recomm_ShortInterest    96.615851
Activism2               98.102903
ProbInformedTrading     99.029333
IO_ShortInterest        99.196073
Length: 211, dtype: float64


In [36]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged = merged[merged['FF49'].notna()].copy()
merged.to_csv(f"{OUTPUT_DIR}/cleaned_merged_data.csv", index=False)
ff_all.to_csv(f"{OUTPUT_DIR}/ff_factors_clean.csv", index=False)


In [37]:
print(merged.head())
print(ff_all.head())

    permno  yyyymm        AM       AOP  AbnormalAccruals  Accruals  \
2    10001  201412  1.762830  0.042705         -0.005713  0.034970   
4    10001  201612  1.497349       NaN         -0.039445  0.063455   
6    10002  199712  3.365251       NaN               NaN -0.010779   
7    10002  199812  3.135689       NaN               NaN  0.033185   
11   10002  200312  4.752564       NaN               NaN -0.011305   

    AccrualsBM  Activism1  Activism2     AdExp  ...  RETURN_YEAR  PERMNO_x  \
2          NaN        NaN        NaN       NaN  ...         2015     10001   
4          NaN        NaN        NaN       NaN  ...         2017     10001   
6          NaN        NaN        NaN  0.001250  ...         1998     10002   
7          NaN        NaN        NaN  0.001425  ...         1999     10002   
11         NaN        NaN        NaN  0.002777  ...         2004     10002   

    YEAR_y  RET_ANNUAL  PERMNO_y    YEAR  PRC_ABS  MARKET_CAP   SICCD  FF49  
2     2015   -0.065084   10001.0

In [38]:
print("Data cleaning complete. ")

Data cleaning complete. 


In [39]:
#3: Z score normalization
print(merged.columns.to_list())
id_cols = ['permno', 'yyyymm', 'YEAR_x', 'MONTH', 'RETURN_YEAR', 'PERMNO_x', 
           'YEAR_y', 'RET_ANNUAL', 'PERMNO_y', 'YEAR', 'PRC_ABS', 'MARKET_CAP', 'SICCD', 'FF49']
factor_cols = [c for c in merged.columns if c not in id_cols]
print(f"Factors to z-score: {len(factor_cols)}")


['permno', 'yyyymm', 'AM', 'AOP', 'AbnormalAccruals', 'Accruals', 'AccrualsBM', 'Activism1', 'Activism2', 'AdExp', 'AgeIPO', 'AnalystRevision', 'AnalystValue', 'AnnouncementReturn', 'AssetGrowth', 'BM', 'BMdec', 'BPEBM', 'Beta', 'BetaFP', 'BetaLiquidityPS', 'BetaTailRisk', 'BidAskSpread', 'BookLeverage', 'BrandInvest', 'CBOperProf', 'CF', 'CPVolSpread', 'Cash', 'CashProd', 'ChAssetTurnover', 'ChEQ', 'ChForecastAccrual', 'ChInv', 'ChInvIA', 'ChNAnalyst', 'ChNNCOA', 'ChNWC', 'ChTax', 'ChangeInRecommendation', 'CitationsRD', 'CompEquIss', 'CompositeDebtIssuance', 'ConsRecomm', 'ConvDebt', 'CoskewACX', 'Coskewness', 'CredRatDG', 'CustomerMomentum', 'DebtIssuance', 'DelBreadth', 'DelCOA', 'DelCOL', 'DelDRC', 'DelEqu', 'DelFINL', 'DelLTI', 'DelNetFin', 'DivInit', 'DivOmit', 'DivSeason', 'DivYieldST', 'DolVol', 'DownRecomm', 'EBM', 'EP', 'EarnSupBig', 'EarningsConsistency', 'EarningsForecastDisparity', 'EarningsStreak', 'EarningsSurprise', 'EntMult', 'EquityDuration', 'ExchSwitch', 'ExclExp',

In [40]:
merged_z = merged.copy()

grouped = merged_z.groupby(['FF49', 'RETURN_YEAR'])[factor_cols]
means = grouped.transform('mean')
stds = grouped.transform('std')

merged_z[factor_cols] = (merged_z[factor_cols] - means) / stds.replace(0, np.nan)
merged_z[factor_cols] = merged_z[factor_cols].fillna(0)

print("Z-scoring complete")

Z-scoring complete


In [41]:
print(f"RET_ANNUAL unchanged: {(merged['RET_ANNUAL'] == merged_z['RET_ANNUAL']).all()}")
print(f"Sample z-scored factor mean: {merged_z['BM'].mean():.4f} (should be ≈0)")
merged_z.to_csv("data/cleaned_merged_zscored.csv", index=False)
merged_z.head()

RET_ANNUAL unchanged: True
Sample z-scored factor mean: -0.0000 (should be ≈0)


,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,...,RETURN_YEAR,PERMNO_x,YEAR_y,RET_ANNUAL,PERMNO_y,YEAR,PRC_ABS,MARKET_CAP,SICCD,FF49
2,10001,201412,0.150356,0.177744,0.092818,0.196376,0.0,0.0,0.0,0.000000,...,2015,10001,2015,-0.065084,10001.0,2014.0,11.02,115577.76,4925.0,31.0
4,10001,201612,-0.272024,0.000000,-0.311768,1.013648,0.0,0.0,0.0,0.000000,...,2017,10001,2017,0.043988,10001.0,2016.0,12.55,132026.00,4925.0,31.0
6,10002,199712,-0.107782,0.000000,0.000000,-0.100255,0.0,0.0,0.0,-0.434780,...,1998,10002,1998,0.014244,10002.0,1997.0,24.50,104027.00,6710.0,48.0
7,10002,199812,-0.339190,0.000000,0.000000,0.448420,0.0,0.0,0.0,-0.499539,...,1999,10002,1999,-0.064491,10002.0,1998.0,15.25,117867.25,6710.0,48.0
11,10002,200312,-0.073657,0.000000,0.000000,-0.192869,0.0,0.0,0.0,-0.337381,...,2004,10002,2004,0.578180,10002.0,2003.0,16.02,140094.90,6020.0,45.0


In [42]:
#4: Fama Macbeth
IN_SAMPLE_START = 1986
IN_SAMPLE_END = 2007

in_sample = merged_z[(merged_z['RETURN_YEAR'] >= IN_SAMPLE_START) & 
                      (merged_z['RETURN_YEAR'] <= IN_SAMPLE_END)]

print(f"In-sample: {len(in_sample):,} rows")
print(f"Years: {in_sample['RETURN_YEAR'].min()} - {in_sample['RETURN_YEAR'].max()}")
print(f"Unique years: {in_sample['RETURN_YEAR'].nunique()}")


In-sample: 59,181 rows
Years: 1986 - 2007
Unique years: 22


In [43]:
import statsmodels.api as sm
from collections import defaultdict
yearly_coefs = defaultdict(list)
for year in sorted(in_sample['RETURN_YEAR'].unique()):
    year_data = in_sample[in_sample['RETURN_YEAR'] == year].copy()
    year_data_clean = year_data[factor_cols + ['RET_ANNUAL']].dropna()
    
    if len(year_data_clean) < 50: 
        continue
    
    X = year_data_clean[factor_cols]
    X = sm.add_constant(X)
    y = year_data_clean['RET_ANNUAL']
    
    try:
        model = sm.OLS(y, X).fit()
        for col in factor_cols:
            yearly_coefs[col].append(model.params.get(col, np.nan))
    except:
        continue
    
    print(f"Year {year}: {len(year_data_clean):,} obs")

print(f"\n✓ Completed {len(yearly_coefs[factor_cols[0]])} years")

Year 1986: 1,833 obs
Year 1987: 1,859 obs
Year 1988: 1,714 obs
Year 1989: 1,806 obs
Year 1990: 1,793 obs
Year 1991: 1,610 obs
Year 1992: 2,049 obs
Year 1993: 2,301 obs
Year 1994: 2,694 obs
Year 1995: 2,737 obs
Year 1996: 3,176 obs
Year 1997: 3,565 obs
Year 1998: 3,794 obs
Year 1999: 3,414 obs
Year 2000: 3,532 obs
Year 2001: 2,953 obs
Year 2002: 2,886 obs
Year 2003: 2,556 obs
Year 2004: 3,121 obs
Year 2005: 3,242 obs
Year 2006: 3,245 obs
Year 2007: 3,301 obs

✓ Completed 22 years


In [44]:
fm_results = []

for col in factor_cols:
    coefs = [c for c in yearly_coefs[col] if not np.isnan(c)]
    if len(coefs) < 5:
        continue
    
    avg_premium = np.mean(coefs)
    std_premium = np.std(coefs, ddof=1)
    t_stat = avg_premium / (std_premium / np.sqrt(len(coefs))) if std_premium > 0 else 0
    
    fm_results.append({
        'Factor': col,
        'Avg_Premium': avg_premium,
        'Std_Premium': std_premium,
        't_stat': t_stat,
        'N_Years': len(coefs)
    })

fm_df = pd.DataFrame(fm_results)
fm_df['abs_t'] = fm_df['t_stat'].abs()
fm_df = fm_df.sort_values('abs_t', ascending=False)

print(f"Factors analyzed: {len(fm_df)}")

Factors analyzed: 209


In [45]:
print("TOP 30 FACTORS BY |t-stat|:\n")
print(fm_df[['Factor', 'Avg_Premium', 't_stat', 'N_Years']].head(30).to_string(index=False))

TOP 30 FACTORS BY |t-stat|:

              Factor  Avg_Premium    t_stat  N_Years
                  RD     0.018232  4.356638       22
              DelEqu     0.018200  4.260728       22
             Spinoff     0.015980  4.178968       22
        BidAskSpread     0.016043  3.599503       22
                Cash     0.010086  3.253055       22
         TrendFactor     0.013002  2.806713       22
            grcapx3y     0.007972  2.779962       22
             GrAdExp     0.011593  2.600235       22
         VolumeTrend     0.009059  2.555406       22
                 AOP     0.007155  2.501018       22
               BPEBM     0.020513  2.345635       22
              AgeIPO    -0.029999 -2.260625       22
  AnnouncementReturn     0.004764  2.249903       22
                hire    -0.006223 -2.239075       22
                 cfp    -0.007217 -2.226040       22
         Illiquidity     0.011434  2.198239       22
            RIO_Disp     0.016495  2.192727       22
          DelBrea

In [46]:
#5: select final model of factors
selected = fm_df[fm_df['abs_t'] >= 1.5].copy()
print(f"Factors with |t| >= 1.5: {len(selected)}")
print(selected[['Factor', 'Avg_Premium', 't_stat']].to_string(index=False))


Factors with |t| >= 1.5: 57
              Factor  Avg_Premium    t_stat
                  RD     0.018232  4.356638
              DelEqu     0.018200  4.260728
             Spinoff     0.015980  4.178968
        BidAskSpread     0.016043  3.599503
                Cash     0.010086  3.253055
         TrendFactor     0.013002  2.806713
            grcapx3y     0.007972  2.779962
             GrAdExp     0.011593  2.600235
         VolumeTrend     0.009059  2.555406
                 AOP     0.007155  2.501018
               BPEBM     0.020513  2.345635
              AgeIPO    -0.029999 -2.260625
  AnnouncementReturn     0.004764  2.249903
                hire    -0.006223 -2.239075
                 cfp    -0.007217 -2.226040
         Illiquidity     0.011434  2.198239
            RIO_Disp     0.016495  2.192727
          DelBreadth     0.009976  2.164166
             IntanEP    -0.035270 -2.161568
                 EBM    -0.018279 -2.110928
              OScore     0.016962  2.107432
    

In [47]:
final_factors = selected.nlargest(40, 'abs_t')[['Factor', 't_stat']]
final_factors.to_csv("data/final_factors.csv", index=False)
print(f"✓ Saved top 40 factors")


✓ Saved top 40 factors


In [48]:
#Out of Sample Scoring:
OOS_START = 2008
OOS_END = 2023

oos_data = merged_z[(merged_z['RETURN_YEAR'] >= OOS_START) & 
                     (merged_z['RETURN_YEAR'] <= OOS_END)]

print(f"Out-of-sample: {len(oos_data):,} rows")
print(f"Years: {oos_data['RETURN_YEAR'].min()} - {oos_data['RETURN_YEAR'].max()}")


Out-of-sample: 42,885 rows
Years: 2008 - 2023


In [49]:
final_factors = pd.read_csv("data/final_factors.csv")
factor_names = final_factors['Factor'].tolist()
factor_weights = dict(zip(final_factors['Factor'], final_factors['t_stat']))

print(f"Scoring with {len(factor_names)} factors")
oos_data = oos_data.copy()
valid_factors = [f for f in factor_names if f in oos_data.columns]
print(f"Valid factors in data: {len(valid_factors)}")
oos_data['SCORE'] = 0
for factor in valid_factors:
    weight = factor_weights[factor]
    oos_data['SCORE'] += oos_data[factor].fillna(0) * weight

print(f"✓ Scored {len(oos_data):,} stock-years")
print(f"Score range: {oos_data['SCORE'].min():.2f} to {oos_data['SCORE'].max():.2f}")


Scoring with 40 factors
Valid factors in data: 40
✓ Scored 42,885 stock-years
Score range: -60.85 to 103.27


In [50]:
oos_data[['permno', 'RETURN_YEAR', 'SCORE', 'RET_ANNUAL']].head(10)

,permno,RETURN_YEAR,SCORE,RET_ANNUAL
2,10001,2015,-8.035746,-0.065084
4,10001,2017,8.043116,0.043988
15,10002,2008,0.193764,0.282612
16,10002,2009,6.487220,-0.552441
64,10025,2008,-3.709663,-0.450797
65,10025,2009,2.068374,1.933332
66,10025,2010,-7.176062,-0.322099
67,10025,2011,8.091149,0.084778
68,10025,2012,12.045550,1.104086
69,10025,2013,8.688757,-0.108052


In [ ]:
#Build Portfolio — explicit decile sort (D10 = top, D1 = bottom) per OOS year
N_BUCKETS = 10

def assign_deciles(group):
    try:
        return pd.qcut(group['SCORE'], N_BUCKETS, labels=False, duplicates='drop') + 1
    except ValueError:
        return pd.Series(np.nan, index=group.index)

oos_data = oos_data.copy()
oos_data['decile'] = (
    oos_data.groupby('RETURN_YEAR', group_keys=False).apply(assign_deciles)
)
oos_data = oos_data.dropna(subset=['decile'])
oos_data['decile'] = oos_data['decile'].astype(int)

portfolio_returns = []
for year in sorted(oos_data['RETURN_YEAR'].unique()):
    yd = oos_data[oos_data['RETURN_YEAR'] == year]
    long_ret = yd.loc[yd['decile'] == N_BUCKETS, 'RET_ANNUAL'].mean()
    short_ret = yd.loc[yd['decile'] == 1, 'RET_ANNUAL'].mean()
    portfolio_returns.append({
        'Year': year,
        'Long_Ret': long_ret,
        'Short_Ret': short_ret,
        'Spread': long_ret - short_ret,
        'N_Long': int((yd['decile'] == N_BUCKETS).sum()),
        'N_Short': int((yd['decile'] == 1).sum()),
    })

port_df = pd.DataFrame(portfolio_returns)
print(port_df.to_string(index=False))
print(f"\n=== ANNUAL D10-D1 PERFORMANCE (OOS {OOS_START}-{OOS_END}) ===")
print(f"Avg Long  (D10) Return: {port_df['Long_Ret'].mean()*100:.2f}%")
print(f"Avg Short (D1)  Return: {port_df['Short_Ret'].mean()*100:.2f}%")
print(f"Avg L-S Spread:         {port_df['Spread'].mean()*100:.2f}%")
print(f"Spread Std Dev:         {port_df['Spread'].std()*100:.2f}%")
print(f"Spread t-stat:          {port_df['Spread'].mean() / (port_df['Spread'].std() / np.sqrt(len(port_df))):.2f}")


In [ ]:
#Monthly long-short series using the decile labels + cumulative equity curve
decile_map = oos_data[['permno', 'RETURN_YEAR', 'decile']]
oos_years = oos_data['RETURN_YEAR'].unique()
monthly = crsp.loc[crsp['YEAR'].isin(oos_years), ['PERMNO', 'YEAR', 'YYYYMM', 'RET']].copy()
joined = monthly.merge(
    decile_map,
    left_on=['PERMNO', 'YEAR'],
    right_on=['permno', 'RETURN_YEAR'],
    how='inner',
)

monthly_dec = (
    joined.groupby(['YYYYMM', 'decile'])['RET'].mean().unstack('decile').sort_index()
)
monthly_dec.columns = [f'D{int(c)}' for c in monthly_dec.columns]

monthly_df = monthly_dec.copy()
monthly_df['Long_Ret'] = monthly_df[f'D{N_BUCKETS}']
monthly_df['Short_Ret'] = monthly_df['D1']
monthly_df['Spread'] = monthly_df['Long_Ret'] - monthly_df['Short_Ret']
monthly_df['Cumulative'] = (1.0 + monthly_df['Spread']).cumprod() - 1.0
monthly_df = monthly_df.reset_index()

print(f"Monthly returns: {len(monthly_df)} months ({monthly_df['YYYYMM'].min()}-{monthly_df['YYYYMM'].max()})")
print(f"Cumulative D10-D1 return: {monthly_df['Cumulative'].iloc[-1]*100:.2f}%")


In [53]:
ff_all = pd.read_csv("data/ff_factors_clean.csv")
monthly_df = pd.merge(monthly_df, ff_all, on='YYYYMM', how='inner')
print(f"Merged: {len(monthly_df)} months")
monthly_df.head()

Merged: 192 months


,YYYYMM,Long_Ret,Short_Ret,Spread,MKT_RF,SMB,HML,RF,UMD
0,200801,-0.067155,-0.066523,-0.000632,-0.0633,-0.0102,0.0400,0.0021,-0.0774
1,200802,-0.029773,-0.047554,0.017781,-0.0309,-0.0048,-0.0072,0.0013,0.0613
2,200803,0.006267,-0.028980,0.035248,-0.0094,0.0071,0.0031,0.0017,0.0425
3,200804,0.052473,0.053260,-0.000786,0.0461,-0.0169,-0.0096,0.0018,-0.0033
4,200805,0.051193,0.057814,-0.006621,0.0187,0.0297,-0.0150,0.0018,0.0339


In [ ]:
import statsmodels.api as sm

reg = monthly_df.merge(ff_all, on='YYYYMM', how='inner').dropna(
    subset=['Spread', 'MKT_RF', 'SMB', 'HML', 'UMD']
)
y = reg['Spread']

X_capm = sm.add_constant(reg[['MKT_RF']])
capm = sm.OLS(y, X_capm).fit()

X_4f = sm.add_constant(reg[['MKT_RF', 'SMB', 'HML', 'UMD']])
ff4 = sm.OLS(y, X_4f).fit()

# Raw stats
raw_mean_ann = y.mean() * 12
raw_std_ann = y.std(ddof=1) * np.sqrt(12)
sharpe = raw_mean_ann / raw_std_ann

alpha_capm_m = capm.params['const']
alpha_4f_m = ff4.params['const']
resid_std_m = ff4.resid.std(ddof=1)
# IR = annualized 4F alpha / annualized residual std = (alpha_m / resid_std_m) * sqrt(12)
IR = (alpha_4f_m / resid_std_m) * np.sqrt(12)

print(f"=== RAW (monthly L/S, annualized) ===")
print(f"Return: {raw_mean_ann*100:.2f}%/yr,  Vol: {raw_std_ann*100:.2f}%/yr,  Sharpe: {sharpe:.2f}")

print(f"\n=== CAPM ===")
print(f"Alpha (annual): {alpha_capm_m*12*100:.2f}%")
print(f"Alpha t-stat:   {capm.tvalues['const']:.2f}")
print(f"Beta_MKT:       {capm.params['MKT_RF']:.2f}")
print(f"R^2:            {capm.rsquared:.3f}")

print(f"\n=== 4-FACTOR (Carhart) ===")
print(f"Alpha (annual): {alpha_4f_m*12*100:.2f}%")
print(f"Alpha t-stat:   {ff4.tvalues['const']:.2f}")
print(f"R^2:            {ff4.rsquared:.3f}")
print("Factor loadings:")
for f in ['MKT_RF', 'SMB', 'HML', 'UMD']:
    print(f"  {f:7s}: {ff4.params[f]:+.3f}  (t={ff4.tvalues[f]:+.2f})")

print(f"\nInformation Ratio (annualized): {IR:.2f}")


In [ ]:
#Out of sample predictions. 